### Principal component analysis (PCA)

1. Load or create a dataset with more than 2 dimensions.
2. Find the first 2 principal components with and without using sklearn.
3. Practice using PCA to preserve a certain percentage of variance.
4. Train a classification or regression neural network by using: 
- a) The Original dataset
- b) Principal components

5. Repeat part 4 using Kernel PCA (linear, sigmoid, RBF).
6. Build a pipeline to tune the hyperparameters of Kernel PCA and also the neural network. Which
hyperparameters can be tuned?

2D means that the dataset has 2 or more featurs/colums per sample

In [ ]:
# lets create some cluster blobs for a 2D dataset

from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA

X,y = make_blobs(n_samples=300, # number of points
                n_features=2,   # the dimension
                centers=4,      # number of cluster centers generated
                cluster_std=1.4,# the spread of the clusters
                random_state=42)# seed



In [ ]:
# PCA principa component analysis linear dimenson-reduction method
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X)
print(pca2.explained_variance_ratio_)
print("total variance",pca2.explained_variance_ratio_.sum())

pca95 = PCA(n_components=0.95) # preserve 95% of the data
X_pca95 = pca95.fit_transform(X)
print("asd",pca95.n_components_)
print("total variance",pca95.explained_variance_ratio_.sum())
print(f"seems like we find {pca95.n_components_} PCs becouse there only are {pca95.n_components_} ")
print("increaseing the amount of diemnsions to 5 will increase the amount of componetns to 3,\n but since we are only using 2 and finding 2 means that we are finding all the diemnsion \nof the data ")

[0.58117062 0.41882938]
total variance 1.0
asd 2
total variance 1.0
seems like we find 2 PCs becouse there only are 2 
increaseing the amount of diemnsions to 5 will increase the amount of componetns to 3,
 but since we are only using 2 and finding 2 means that we are finding all the diemnsion 
of the data 


In [22]:
import numpy as np

# center data
X_centered = X - X.mean(axis=0)

# SVD on covariance (or directly on centered data)
U, S, Vt = np.linalg.svd(X_centered / np.sqrt(X_centered.shape[0]-1), full_matrices=False)
# columns of V = principal axes; S^2 = variances along PCs
explained_variance = S**2
explained_variance_ratio = explained_variance / explained_variance.sum()

# first 2 PCs and projections
components_2 = Vt[:2]             # shape (2, n_features)
scores_2 = X_centered @ components_2.T  # projected data

print("Explained variance ratio (manual):", explained_variance_ratio[:5])
print("Total variance captured by first 2:", explained_variance_ratio[:2].sum())

# variance threshold example (95%)
cum = explained_variance_ratio.cumsum()
k95 = np.searchsorted(cum, 0.95) + 1
components_95 = Vt[:k95]
scores_95 = X_centered @ components_95.T
print("PCs needed for 95% variance (manual):", k95)

Explained variance ratio (manual): [0.58117062 0.41882938]
Total variance captured by first 2: 1.0
PCs needed for 95% variance (manual): 2


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# split the data simillar to intro course
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# normalize it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# A ) train on the original dataset
model_OG = Sequential([
    Dense(16, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dense(8,activation='relu'),
    Dense(4,activation='softmax'),
])
model_OG.compile(optimizer=Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
hist_OG = model_OG.fit(X_train_scaled,y_train,epochs=50,batch_size=32,validation_split=0.2,verbose=2)




Epoch 1/50
6/6 - 0s - loss: 1.2931 - accuracy: 0.4896 - val_loss: 1.2798 - val_accuracy: 0.4583
Epoch 2/50
6/6 - 0s - loss: 1.2632 - accuracy: 0.5052 - val_loss: 1.2534 - val_accuracy: 0.4583
Epoch 3/50
6/6 - 0s - loss: 1.2365 - accuracy: 0.5052 - val_loss: 1.2298 - val_accuracy: 0.4583
Epoch 4/50
6/6 - 0s - loss: 1.2109 - accuracy: 0.5052 - val_loss: 1.2075 - val_accuracy: 0.4583
Epoch 5/50
6/6 - 0s - loss: 1.1868 - accuracy: 0.5052 - val_loss: 1.1858 - val_accuracy: 0.4583
Epoch 6/50
6/6 - 0s - loss: 1.1626 - accuracy: 0.5052 - val_loss: 1.1653 - val_accuracy: 0.4583
Epoch 7/50
6/6 - 0s - loss: 1.1408 - accuracy: 0.5052 - val_loss: 1.1454 - val_accuracy: 0.4583
Epoch 8/50
6/6 - 0s - loss: 1.1187 - accuracy: 0.5052 - val_loss: 1.1262 - val_accuracy: 0.4583
Epoch 9/50
6/6 - 0s - loss: 1.0986 - accuracy: 0.5052 - val_loss: 1.1075 - val_accuracy: 0.4583
Epoch 10/50
6/6 - 0s - loss: 1.0775 - accuracy: 0.5052 - val_loss: 1.0898 - val_accuracy: 0.4583
Epoch 11/50
6/6 - 0s - loss: 1.0589 - a

In [ ]:
# B )
pca_train = PCA(n_components=2)
X_train_PCA = pca_train.fit_transform(X_train_scaled)
X_test_PCA = pca_train.transform(X_test_scaled)

model_PCA = Sequential([
      Dense(16, activation='relu', input_dim=2),
    Dense(8,activation='relu'),
    Dense(4,activation='softmax'),
])

model_PCA.compile(optimizer=Adam(learning_rate=0.0001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

hist_PCA = model_PCA.fit(X_train_PCA,y_train,
                        epochs=50,
                        batch_size=32,
                        validation_split=.02,
                        verbose=2)

Classification problem
1. Load IRIS and MNIST fashion datasets from Keras.
2. Build and train a neural network for classifying the loaded labeled datasets.
3. Tune the hyperparameters (including hidden layer size and activation functions). What else can be tuned?
4. Plot the loss and accuracy for training and testing datasets.
5. Save the weights of the layers and use callbacks during the training process.
6. Practice saving and loading the trained model.

Regression problem
1. Load California housing dataset and split it to training and testing datasets.
2. Build and train a neural network for price prediction. Tune the hyperparameters and discuss the results.
3. Plot the network history.
4. Change the learning rate. Train the network again and plot the network history. Discuss the results.
5. Save the weights of the layers and use callbacks during the training process.
6. Practice saving and loading the trained model.